<!-- MUTERBANDUNG_NOTEBOOK_STATUS_V1 -->
# Notebook Status - Housekeeping Notebook Wisata

| Item | Isi |
|---|---|
| Status | HOUSEKEEPING |
| Fungsi | Audit status notebook dan pembuatan index notebook wisata. |
| Input utama | File `.ipynb` di `Wisata_Workspace/02_Notebooks`. |
| Output utama | `NOTEBOOK_INDEX_WISATA_2026-06-08.md` dan tabel status notebook. |
| Aturan pakai | Boleh dijalankan ulang untuk cek struktur notebook. Tidak menjalankan pipeline data lama. |
| Catatan | Notebook ini hanya merapikan rujukan kerja, bukan memproses dataset wisata. |

---


# Wisata Notebook Housekeeping - 2026-06-08

Notebook ini hanya untuk merapikan status notebook wisata. Tidak ada dataset yang diubah, tidak ada model yang dilatih, dan notebook lama tidak dijalankan ulang.

Pola catatan mengikuti format yang dipakai di training penginapan: tujuan kecil -> code/output -> keputusan singkat.


## Tahap 0 - Menetapkan Folder Notebook

Tujuan: memastikan audit membaca folder notebook wisata yang benar.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 100)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Wisata_Workspace").exists():
            return candidate
    raise FileNotFoundError("Wisata_Workspace tidak ditemukan.")

PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "Wisata_Workspace" / "02_Notebooks"
ARCHIVE_DIR = NOTEBOOK_DIR / "archive"
INDEX_PATH = NOTEBOOK_DIR / "NOTEBOOK_INDEX_WISATA_2026-06-08.md"
HOUSEKEEPING_NOTEBOOK = "notebook_housekeeping_wisata_2026-06-08.ipynb"

setup_df = pd.DataFrame([
    {"item": "project_root", "path": PROJECT_ROOT.as_posix(), "exists": PROJECT_ROOT.exists()},
    {"item": "notebook_dir", "path": NOTEBOOK_DIR.relative_to(PROJECT_ROOT).as_posix(), "exists": NOTEBOOK_DIR.exists()},
    {"item": "archive_dir", "path": ARCHIVE_DIR.relative_to(PROJECT_ROOT).as_posix(), "exists": ARCHIVE_DIR.exists()},
])
display(setup_df)


,item,path,exists
0,project_root,D:/File/file/Fauzan Lubada/PIJAK,True
1,notebook_dir,Wisata_Workspace/02_Notebooks,True
2,archive_dir,Wisata_Workspace/02_Notebooks/archive,True


### Keputusan: folder notebook wisata valid

Housekeeping dilakukan di `Wisata_Workspace/02_Notebooks`. Folder `archive` dipakai sebagai tempat backup/riwayat, bukan sebagai jalur kerja utama.


## Fase A - Inventaris Notebook

Tujuan: melihat semua notebook wisata yang ada saat ini tanpa menjalankan isinya.


In [2]:
notebook_files = sorted(NOTEBOOK_DIR.glob("*.ipynb"))

inventory_rows = []
for path in notebook_files:
    stat = path.stat()
    inventory_rows.append({
        "notebook": path.name,
        "size_kb": round(stat.st_size / 1024, 2),
        "last_modified": pd.to_datetime(stat.st_mtime, unit="s"),
    })

inventory_df = pd.DataFrame(inventory_rows).sort_values("notebook").reset_index(drop=True)
display(inventory_df)

archive_rows = []
if ARCHIVE_DIR.exists():
    for path in sorted(ARCHIVE_DIR.glob("*.ipynb")):
        stat = path.stat()
        archive_rows.append({
            "archive_notebook": path.name,
            "size_kb": round(stat.st_size / 1024, 2),
            "last_modified": pd.to_datetime(stat.st_mtime, unit="s"),
        })
archive_df = pd.DataFrame(archive_rows)
display(archive_df)


,notebook,size_kb,last_modified
0,Data_Completion_Google_Colab.ipynb,15.77,2026-06-08 11:56:16.030809641
1,IndoBERT_Colab.ipynb,5.14,2026-06-08 11:56:16.037361622
2,Media_Fill_Google_Colab.ipynb,16.87,2026-06-08 11:56:16.026836872
3,NLP_Sentiment_Preparation.ipynb,5.77,2026-06-08 11:56:16.020801306
4,Penggabungan_Master_Reviews.ipynb,4.59,2026-06-08 11:56:16.022858381
5,Realworld_Verification_Google_Colab.ipynb,13.01,2026-06-08 11:56:16.034855604
6,Wisata_Training_1_Audit_Master_Data.ipynb,4.86,2026-06-08 11:56:16.039424181
7,notebook_housekeeping_wisata_2026-06-08.ipynb,49.31,2026-06-08 11:58:04.870171309
8,wisata_training.ipynb,260.96,2026-06-08 11:56:15.941543818
9,wisata_traning.ipynb,392.83,2026-06-08 11:56:16.100967884


,archive_notebook,size_kb,last_modified
0,wisata_training_after_phase_a_c_2026-06-06.ipynb,53.15,2026-06-06 13:33:28.818965673
1,wisata_training_after_phase_a_f_2026-06-08.ipynb,133.40,2026-06-06 13:52:54.280033827
2,wisata_training_before_decision_driven_2026-06-06.ipynb,35.25,2026-05-29 06:19:02.801265478


### Keputusan: semua notebook dicatat dulu, belum dipindahkan

Notebook lama dan notebook Colab tetap dibiarkan di tempatnya. Pada tahap housekeeping ini, statusnya diperjelas lewat index agar tidak salah dipakai sebagai pipeline utama.


## Fase B - Klasifikasi Status Notebook

Tujuan: menetapkan notebook mana yang utama, pendukung, Colab, atau legacy/arsip. Keputusan ini tidak menghapus file apa pun.


In [3]:
status_map = {
    "wisata_training.ipynb": {
        "status": "utama",
        "fungsi": "Decision log utama untuk audit dan kesiapan dataset wisata.",
        "aturan_pakai": "Boleh dilanjutkan bertahap. Jangan loncat langsung ke LLM.",
    },
    HOUSEKEEPING_NOTEBOOK: {
        "status": "housekeeping",
        "fungsi": "Index dan audit status notebook wisata.",
        "aturan_pakai": "Boleh dijalankan ulang untuk cek struktur notebook.",
    },
    "NLP_Sentiment_Preparation.ipynb": {
        "status": "pendukung_sentiment",
        "fungsi": "Persiapan/penggabungan data review untuk sentiment.",
        "aturan_pakai": "Jangan dijalankan sebelum path dan input diverifikasi.",
    },
    "IndoBERT_Colab.ipynb": {
        "status": "colab_model_berat",
        "fungsi": "Notebook training/evaluasi model berat di Colab.",
        "aturan_pakai": "Jalankan di Colab/GPU, bukan sebagai pipeline lokal utama.",
    },
    "Media_Fill_Google_Colab.ipynb": {
        "status": "pendukung_enrichment",
        "fungsi": "Workflow pengisian media/gambar dari Colab.",
        "aturan_pakai": "Dipakai saat perlu enrichment media, bukan ranking utama.",
    },
    "Data_Completion_Google_Colab.ipynb": {
        "status": "pendukung_completion",
        "fungsi": "Workflow completion manual/Colab untuk data wisata.",
        "aturan_pakai": "Dipakai saat ada batch completion yang jelas.",
    },
    "Realworld_Verification_Google_Colab.ipynb": {
        "status": "pendukung_verifikasi",
        "fungsi": "Verifikasi real-world untuk lokasi yang perlu dicek.",
        "aturan_pakai": "Jalankan hanya untuk queue verifikasi, bukan semua data.",
    },
    "Penggabungan_Master_Reviews.ipynb": {
        "status": "pendukung_reviews",
        "fungsi": "Penggabungan review mentah menjadi master review.",
        "aturan_pakai": "Jangan rerun sebelum input review dikunci.",
    },
    "Wisata_Training_1_Audit_Master_Data.ipynb": {
        "status": "legacy_ringkas",
        "fungsi": "Audit awal lama untuk master data geospatial.",
        "aturan_pakai": "Referensi historis; bukan notebook utama.",
    },
    "wisata_traning.ipynb": {
        "status": "legacy_arsip",
        "fungsi": "Notebook lama/eksperimen dengan typo nama dan path Colab lama.",
        "aturan_pakai": "Jangan dijalankan sebagai pipeline utama.",
    },
}

classified_rows = []
for _, row in inventory_df.iterrows():
    info = status_map.get(row["notebook"], {
        "status": "belum_diklasifikasi",
        "fungsi": "Belum ada keputusan.",
        "aturan_pakai": "Cek manual sebelum dipakai.",
    })
    classified_rows.append({
        "notebook": row["notebook"],
        "status": info["status"],
        "fungsi": info["fungsi"],
        "aturan_pakai": info["aturan_pakai"],
    })

classified_df = pd.DataFrame(classified_rows).sort_values(["status", "notebook"]).reset_index(drop=True)
display(classified_df)


,notebook,status,fungsi,aturan_pakai
0,IndoBERT_Colab.ipynb,colab_model_berat,Notebook training/evaluasi model berat di Colab.,"Jalankan di Colab/GPU, bukan sebagai pipeline lokal utama."
1,notebook_housekeeping_wisata_2026-06-08.ipynb,housekeeping,Index dan audit status notebook wisata.,Boleh dijalankan ulang untuk cek struktur notebook.
2,wisata_traning.ipynb,legacy_arsip,Notebook lama/eksperimen dengan typo nama dan path Colab lama.,Jangan dijalankan sebagai pipeline utama.
3,Wisata_Training_1_Audit_Master_Data.ipynb,legacy_ringkas,Audit awal lama untuk master data geospatial.,Referensi historis; bukan notebook utama.
4,Data_Completion_Google_Colab.ipynb,pendukung_completion,Workflow completion manual/Colab untuk data wisata.,Dipakai saat ada batch completion yang jelas.
5,Media_Fill_Google_Colab.ipynb,pendukung_enrichment,Workflow pengisian media/gambar dari Colab.,"Dipakai saat perlu enrichment media, bukan ranking utama."
6,Penggabungan_Master_Reviews.ipynb,pendukung_reviews,Penggabungan review mentah menjadi master review.,Jangan rerun sebelum input review dikunci.
7,NLP_Sentiment_Preparation.ipynb,pendukung_sentiment,Persiapan/penggabungan data review untuk sentiment.,Jangan dijalankan sebelum path dan input diverifikasi.
8,Realworld_Verification_Google_Colab.ipynb,pendukung_verifikasi,Verifikasi real-world untuk lokasi yang perlu dicek.,"Jalankan hanya untuk queue verifikasi, bukan semua data."
9,wisata_training.ipynb,utama,Decision log utama untuk audit dan kesiapan dataset wisata.,Boleh dilanjutkan bertahap. Jangan loncat langsung ke LLM.


### Keputusan: `wisata_training.ipynb` tetap menjadi notebook utama

Notebook lain dipakai sebagai pendukung atau arsip sesuai fungsi. Notebook legacy tidak dihapus, tetapi tidak boleh menjadi rujukan utama untuk proses baru.


## Fase C - Audit Struktur Notebook

Tujuan: membaca jumlah cell, output, error lama, dan tanda path Colab/root lama. Ini membantu tahu notebook mana yang aman dibuka dan mana yang perlu hati-hati.


In [4]:
def read_notebook_safely(path):
    try:
        return json.loads(path.read_text(encoding="utf-8")), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"

def first_heading(cells):
    for cell in cells:
        if cell.get("cell_type") == "markdown":
            source = "".join(cell.get("source", []))
            for line in source.splitlines():
                if line.strip().startswith("#"):
                    return line.strip()[:120]
    return ""

structure_rows = []
for path in notebook_files:
    nb_json, err = read_notebook_safely(path)
    if err:
        structure_rows.append({
            "notebook": path.name,
            "read_ok": False,
            "cells_total": None,
            "markdown_cells": None,
            "code_cells": None,
            "code_with_outputs": None,
            "error_outputs": None,
            "has_colab_path": None,
            "has_legacy_dataset_path": None,
            "first_heading": err,
        })
        continue
    nb_cells = nb_json.get("cells", [])
    code_cells = [c for c in nb_cells if c.get("cell_type") == "code"]
    source_all = "\n".join("".join(c.get("source", [])) for c in nb_cells)
    error_outputs = 0
    for cell in code_cells:
        for out in cell.get("outputs", []):
            if out.get("output_type") == "error":
                error_outputs += 1
    is_housekeeping_self = path.name == HOUSEKEEPING_NOTEBOOK
    structure_rows.append({
        "notebook": path.name,
        "read_ok": True,
        "cells_total": len(nb_cells),
        "markdown_cells": sum(1 for c in nb_cells if c.get("cell_type") == "markdown"),
        "code_cells": len(code_cells),
        "code_with_outputs": sum(1 for c in code_cells if c.get("outputs")),
        "error_outputs": error_outputs,
        "has_colab_path": False if is_housekeeping_self else ("/content/drive" in source_all),
        "has_legacy_dataset_path": False if is_housekeeping_self else ("Dataset/" in source_all or "../Dataset" in source_all),
        "first_heading": first_heading(nb_cells),
    })

structure_df = pd.DataFrame(structure_rows).sort_values("notebook").reset_index(drop=True)
display(structure_df)

risk_view = structure_df[(structure_df["error_outputs"].fillna(0).astype(int) > 0) | (structure_df["has_colab_path"].eq(True)) | (structure_df["has_legacy_dataset_path"].eq(True))]
display(risk_view[["notebook", "error_outputs", "has_colab_path", "has_legacy_dataset_path", "first_heading"]])


,notebook,read_ok,cells_total,markdown_cells,code_cells,code_with_outputs,error_outputs,has_colab_path,has_legacy_dataset_path,first_heading
0,Data_Completion_Google_Colab.ipynb,True,19,10,9,0,0,False,True,# Notebook Status - Pendukung Data Completion
1,IndoBERT_Colab.ipynb,True,6,2,4,0,0,False,False,# Notebook Status - Colab Model Berat
2,Media_Fill_Google_Colab.ipynb,True,19,10,9,0,0,False,True,# Notebook Status - Pendukung Media Enrichment
3,NLP_Sentiment_Preparation.ipynb,True,9,5,4,0,0,False,True,# Notebook Status - Pendukung Sentiment
4,Penggabungan_Master_Reviews.ipynb,True,8,2,6,0,0,False,True,# Notebook Status - Pendukung Review Merge
5,Realworld_Verification_Google_Colab.ipynb,True,19,10,9,0,0,False,True,# Notebook Status - Pendukung Real-World Verification
6,Wisata_Training_1_Audit_Master_Data.ipynb,True,7,3,4,0,0,False,False,# Notebook Status - Legacy Audit Ringkas
7,notebook_housekeeping_wisata_2026-06-08.ipynb,True,18,13,5,5,0,False,False,# Notebook Status - Housekeeping Notebook Wisata
8,wisata_training.ipynb,True,33,23,10,10,0,False,False,# Notebook Status - Wisata Training Utama
9,wisata_traning.ipynb,True,60,17,43,25,1,True,True,# Notebook Status - Legacy/Arsip


,notebook,error_outputs,has_colab_path,has_legacy_dataset_path,first_heading
0,Data_Completion_Google_Colab.ipynb,0,False,True,# Notebook Status - Pendukung Data Completion
2,Media_Fill_Google_Colab.ipynb,0,False,True,# Notebook Status - Pendukung Media Enrichment
3,NLP_Sentiment_Preparation.ipynb,0,False,True,# Notebook Status - Pendukung Sentiment
4,Penggabungan_Master_Reviews.ipynb,0,False,True,# Notebook Status - Pendukung Review Merge
5,Realworld_Verification_Google_Colab.ipynb,0,False,True,# Notebook Status - Pendukung Real-World Verification
9,wisata_traning.ipynb,1,True,True,# Notebook Status - Legacy/Arsip


### Keputusan: notebook lama tidak perlu dirapikan dengan rerun paksa

Jika notebook punya path Colab, path legacy, atau error output lama, statusnya cukup ditandai. Memaksa rerun notebook lama justru berisiko membuat hasil lama berubah tanpa kebutuhan jelas.


## Fase D - Membuat Index Notebook

Tujuan: membuat file index yang bisa dibaca manusia dan AI agent lain agar tahu notebook mana yang harus dipakai.


In [5]:
index_lines = []
index_lines.append("# Notebook Index Wisata - 2026-06-08")
index_lines.append("")
index_lines.append("Index ini dibuat untuk housekeeping notebook wisata. File ini tidak mengubah dataset dan tidak menggantikan notebook utama.")
index_lines.append("")
index_lines.append("## Aturan Utama")
index_lines.append("")
index_lines.append("- Notebook utama wisata adalah `wisata_training.ipynb`.")
index_lines.append("- Notebook legacy/Colab jangan dijalankan ulang sebelum input dan path diverifikasi.")
index_lines.append("- Notebook pendukung dipakai sesuai kebutuhan, bukan sebagai pipeline utama.")
index_lines.append("- Perubahan proses penting sebaiknya dicatat dengan pola: tujuan -> code/output -> keputusan.")
index_lines.append("")
index_lines.append("## Status Notebook")
index_lines.append("")
index_lines.append("| Notebook | Status | Fungsi | Aturan Pakai |")
index_lines.append("|---|---|---|---|")
for _, row in classified_df.sort_values("notebook").iterrows():
    index_lines.append(f"| `{row['notebook']}` | {row['status']} | {row['fungsi']} | {row['aturan_pakai']} |")
index_lines.append("")
index_lines.append("## Audit Struktur")
index_lines.append("")
index_lines.append("| Notebook | Cells | Markdown | Code | Code Dengan Output | Error Output | Colab Path | Legacy Dataset Path |")
index_lines.append("|---|---:|---:|---:|---:|---:|---|---|")
for _, row in structure_df.sort_values("notebook").iterrows():
    index_lines.append(
        f"| `{row['notebook']}` | {row['cells_total']} | {row['markdown_cells']} | {row['code_cells']} | {row['code_with_outputs']} | {row['error_outputs']} | {row['has_colab_path']} | {row['has_legacy_dataset_path']} |"
    )
index_lines.append("")
index_lines.append("## Keputusan Housekeeping")
index_lines.append("")
index_lines.append("- `wisata_training.ipynb` dipakai sebagai decision log utama.")
index_lines.append("- `notebook_housekeeping_wisata_2026-06-08.ipynb` dipakai untuk audit status notebook.")
index_lines.append("- `wisata_traning.ipynb` tetap disimpan sebagai legacy/arsip karena typo nama, path lama, dan riwayat eksperimen.")
index_lines.append("- Notebook Colab tetap dipertahankan sebagai pendukung proses berat atau enrichment.")
index_lines.append("- Tidak ada notebook lama yang dihapus pada housekeeping ini.")
index_lines.append("")

INDEX_PATH.write_text("\n".join(index_lines), encoding="utf-8")
print("INDEX_PATH:", INDEX_PATH.relative_to(PROJECT_ROOT).as_posix())
print("rows_status:", len(classified_df))
print("rows_structure:", len(structure_df))


INDEX_PATH: Wisata_Workspace/02_Notebooks/NOTEBOOK_INDEX_WISATA_2026-06-08.md
rows_status: 10
rows_structure: 10


### Keputusan: index notebook dibuat sebagai rujukan kerja

Index ini menjadi peta ringkas agar pembaca tidak bingung memilih notebook. Untuk perubahan proses data berikutnya, tetap gunakan `wisata_training.ipynb` sebagai jalur utama.


## Ringkasan Housekeeping

| Area | Keputusan |
|---|---|
| Notebook utama | `wisata_training.ipynb` |
| Notebook housekeeping | `notebook_housekeeping_wisata_2026-06-08.ipynb` |
| Notebook legacy | Disimpan, tidak dihapus |
| Notebook Colab | Pendukung, jangan dipaksa jalan lokal |
| Dataset | Tidak diubah |
| Pipeline/model | Tidak dijalankan |

Housekeeping ini hanya merapikan status dan rujukan kerja notebook.
